# Structured Output and Validation (Managed Agents API)

A **landmark-architecture metadata extractor**. Feed the agent a messy blurb about a famous building like the _Sydney Opera House_ or _Fallingwater_ and get back clean JSON: `title`, `year`, `architect`, `country`, `notes`.

> **CCA-F Domain 4 - Prompts / Structured Output (20%).** The exam trap: do not blindly trust the first JSON. Two disciplines earn the points. First, the **null-if-not-stated** rule - any field absent from the source is `null`, never a hallucinated guess. Second, a **validate-then-retry-once** guard - if `json.loads` fails or a required field is missing, send one corrective follow-up turn instead of shipping garbage.

_Tutorial notebook. These cells make **live, billable, beta-gated** calls. Runs top-to-bottom against the Managed Agents beta._

### 1. Install dependencies

In [1]:
# uv-managed venvs ship WITHOUT pip, so `%pip install` fails here.
# Use uv instead, pointed at THIS kernel's interpreter so packages land
# where the kernel actually looks. Idempotent: uv no-ops if already satisfied.
import subprocess, sys

subprocess.run(
    ["uv", "pip", "install", "--python", sys.executable, "anthropic", "python-dotenv"],
    check=True,
)
print("Dependencies ready.")

Dependencies ready.


### 2. Load environment variables from .env file

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

### 3. Create an API client

The Managed Agents surface lives under `client.beta`. Every call carries the **beta header** via `betas=[BETA]`. Same key that authenticates `messages.create()` authenticates the full agents surface.

In [3]:
import os
from anthropic import Anthropic

client = Anthropic()
MODEL = "claude-haiku-4-5"           # repo default; Haiku handles structured extraction well
BETA = "managed-agents-2026-04-01"   # Managed Agents beta gate

# Teardown guard. Default OFF so "Run All" in class leaves the agent and session
# live for Console inspection. Headless smoke tests set ARCHIVE_ON_RUN=1 to force
# cleanup so no billable resource dangles in CI. See the teardown() in the final cell.
RUN_TEARDOWN = os.environ.get("ARCHIVE_ON_RUN", "0") == "1"

### 4. Create the agent, session, and a send-and-collect helper

The **system prompt** is where structured output is won. It pins the exact schema, forbids prose around the JSON, and states the **null-if-not-stated** rule in one blunt sentence. That last rule is the anti-hallucination lever: a director the blurb never names must come back as `null`, not a plausible guess.

The spine is always the same five moves: **create agent -> create environment -> create session -> send a user turn -> stream events until `session.status_idle`**. The helper below wraps send + stream and reads the agent's text from `agent.message` events, so each teaching turn stays one line.

In [4]:
SYSTEM = """You are a metadata extractor for a landmark-architecture archive.
Return ONLY a single JSON object, no prose, no markdown fences. Exact schema:
{"title": str, "year": int, "architect": str, "country": str, "notes": str}
Rule: any field NOT stated in the source text must be null. Never guess or infer
a value that is not present. Null is the correct answer for missing data."""

REQUIRED_FIELDS = ("title", "year", "architect", "country", "notes")

# create agent -> environment -> session (the spine)
agent = client.beta.agents.create(
    model={"id": MODEL},
    name="architecture-metadata-extractor",
    system=SYSTEM,
    betas=[BETA],
)
env = client.beta.environments.create(name="extractor-scratch", betas=[BETA])
session = client.beta.sessions.create(
    agent=agent.id,
    environment_id=env.id,
    betas=[BETA],
)


def send_and_collect(text: str) -> str:
    """Send one user turn, stream until idle, return the agent's concatenated text.

    Reads only `agent.message` events for text; stops on `session.status_idle`,
    which is the turn-complete signal (the exam point: stop on status, not vibes).
    The agent.message event carries its text blocks directly on `event.content`.
    """
    client.beta.sessions.events.send(
        session.id,
        events=[{"type": "user.message",
                 "content": [{"type": "text", "text": text}]}],
        betas=[BETA],
    )
    chunks = []
    for event in client.beta.sessions.events.stream(session.id, betas=[BETA]):
        if event.type == "agent.message":
            for block in event.content:
                if getattr(block, "type", None) == "text":
                    chunks.append(block.text)
        elif event.type == "session.status_idle":
            break
    return "".join(chunks).strip()


print(f"agent   {agent.id}")
print(f"session {session.id}")

agent   agent_01W3uBMBDLxesFBSMAowV1Pd
session sesn_016hQtf3JrgaWCyRGHKoerG3


### 5. Extract, then validate-and-retry once

Here is the messy blurb. It names the **title** and **year**, states the **country**, but pointedly says nobody knows who the **architect** was. That last gap is the trap: a weak extractor fills `architect` with a famous name that "fits." The null-if-not-stated rule requires `architect` to come back `null`.

The guard runs **two checks** on the returned text and, on failure, sends **exactly one** corrective follow-up turn. The session keeps its memory server-side, so the retry sees the original blurb without re-sending it.

| Validation outcome | What happened | Action |
|---|---|---|
| **Valid** | `json.loads` succeeds, all required keys present | Accept, done in one turn |
| **Parse error** | `json.loads` raises (fences, trailing prose) | One corrective turn: "JSON only, no prose" |
| **Missing field** | Parses, but a required key is absent | One corrective turn naming the missing key |
| **Still bad after retry** | Second turn also fails | Stop, surface the raw text, do not loop forever |

> **Gotcha.** A retry loop with no cap is a billing and latency hazard. **Retry once, then stop.** And never "fix" a missing value by inventing one - a `null` you can see beats a fabrication you cannot.

In [5]:
import json, time

# Note: this blurb deliberately does NOT name the architect. That absence is the
# whole point - the null-if-not-stated rule must return architect=null, not a guess.
BLURB = (
    "finally visited the SYDNEY OPERA HOUSE here in australia, finished in 1973. "
    "those sail-shaped concrete shells over the harbour are unreal, and the concert "
    "hall acoustics are worth the trip alone. no idea who actually designed it, but "
    "the whole structure looks like it grew straight out of the water."
)


def validate(text: str):
    """Return (parsed_dict, error_reason). error_reason is None when the JSON is good."""
    try:
        data = json.loads(text)
    except json.JSONDecodeError as exc:
        return None, f"not valid JSON: {exc.msg}"
    missing = [f for f in REQUIRED_FIELDS if f not in data]
    if missing:
        return None, f"missing required field(s): {', '.join(missing)}"
    return data, None


def extract_with_retry(blurb: str) -> dict:
    """Extract once, validate, and send at most ONE corrective follow-up turn."""
    raw = send_and_collect(f"Extract metadata from this blurb:\n\n{blurb}")
    data, reason = validate(raw)
    if reason is None:
        print("valid on first turn")
        return data

    # one corrective turn - the session already holds the blurb server-side
    print(f"first turn failed ({reason}); retrying once")
    raw = send_and_collect(
        f"Your previous reply was invalid: {reason}. "
        "Return ONLY the JSON object with all five keys. "
        "Any field the source did not state must be null."
    )
    data, reason = validate(raw)
    if reason is None:
        print("valid after one retry")
        return data

    # do NOT loop forever - surface the raw text and stop
    raise ValueError(f"still invalid after one retry ({reason}); raw:\n{raw}")


def teardown():
    """Archive the session and agent - but only when RUN_TEARDOWN is armed.

    Default OFF so "Run All" in class leaves the agent and session LIVE for Console
    inspection; smoke tests set ARCHIVE_ON_RUN=1 to force cleanup. A session can only
    be archived once it is `idle` or `terminated`, so we poll its status briefly first
    - archiving a still-`running` session returns a 400. Bounded wait, never infinite.
    """
    if not RUN_TEARDOWN:
        print("Teardown SKIPPED - agent and session are still LIVE.")
        print("Archive: set ARCHIVE_ON_RUN=1 (or flip RUN_TEARDOWN) and re-run.")
        print(f"  agent:   {agent.id}")
        print(f"  session: {session.id}")
        return
    for _ in range(10):
        status = client.beta.sessions.retrieve(session.id, betas=[BETA]).status
        if status in ("idle", "terminated"):
            break
        time.sleep(1)
    client.beta.sessions.archive(session.id, betas=[BETA])
    client.beta.agents.archive(agent.id, betas=[BETA])
    print("torn down: session and agent archived.")


try:
    result = extract_with_retry(BLURB)
    print(json.dumps(result, indent=2))
    # the null-if-not-stated proof: the architect was never named in the blurb
    assert result["architect"] is None, "architect should be null - the blurb never named who designed it"
    print("\nnull-if-not-stated held: architect is null, not a hallucinated name.")
finally:
    # ALWAYS call teardown - it archives only when armed, else reports live IDs
    teardown()

valid on first turn
{
  "title": "Sydney Opera House",
  "year": 1973,
  "architect": null,
  "country": "Australia",
  "notes": "Sail-shaped concrete shells over the harbour; concert hall acoustics notable"
}

null-if-not-stated held: architect is null, not a hallucinated name.


torn down: session and agent archived.
